# Проверка гипотез и анализ A/B-теста: Яндекс Книги и интернет-магазин BitMotion Kit


# Часть 1. Проверка гипотезы в Python и составление аналитической записки

- Автор: Петрунин Богдан
- Дата: 16.09.2026

## Цели и задачи проекта

**Цель:** статистически проверить гипотезу о том, что пользователи из Санкт-Петербурга в среднем проводят больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы, и оформить результат в виде аналитической записки.

**Задачи:**
- загрузить данные и проверить их на пропуски и дубликаты;
- сравнить размеры групп, их описательные статистики и распределения;
- выбрать подходящий статистический тест и уровень значимости;
- проверить гипотезу и рассчитать p-value;
- сделать вывод по результатам теста и предложить возможные объяснения полученного результата.

## Описание данных

Файл `yandex_knigi_data.csv` содержит данные о пользователях Яндекс Книг из Москвы и Санкт-Петербурга:

- `city` — город пользователя;
- `puid` — идентификатор пользователя;
- `hours` — суммарное количество часов активности (чтения и прослушивания), рассчитанное в SQL по полю `hours`.

## Содержимое проекта

**Часть 1. Яндекс Книги**
1. Загрузка данных и знакомство с ними
2. Проверка гипотезы в Python
3. Аналитическая записка

**Часть 2. A/B-тест интерфейса интернет-магазина BitMotion Kit**
1. Цели и план работ
2. Загрузка данных и проверка целостности
3. Проверка корректности проведения теста
4. Оценка результатов A/B-тестирования
5. Общий вывод

##  Загрузка данных и знакомство с ними

Загрузим данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `data/yandex_knigi_data.csv`.

In [1]:
# Загружаем необходимые библиотеки 
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.power import zt_ind_solve_power
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# Загружаем данные в датафрейм df
df = pd.read_csv('data/yandex_knigi_data.csv',index_col=0)

In [3]:
# Смотрим на датасет
print(df.info())
display(df.head())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8784 entries, 0 to 8783
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   city    8784 non-null   object 
 1   puid    8784 non-null   int64  
 2   hours   8784 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 274.5+ KB
None


,city,puid,hours
0,Москва,9668,26.167776
1,Москва,16598,82.111217
2,Москва,80401,4.656906
3,Москва,140205,1.840556
4,Москва,248755,151.326434


In [4]:
# Проверим наличие полных дубликатов
df.duplicated().sum()  

0

In [5]:
# Проверим дубликаты по пользователям
df['puid'].duplicated().sum()

244

Мы обнаружили, что 244 пользователей встречаются по несколько раз. Проверим подробнее: 

In [6]:
dup_puids = df.loc[df['puid'].duplicated(keep=False), 'puid'].unique()
print(f'Пользователей, встречающихся в обеих группах: {len(dup_puids)} '
      f'({len(dup_puids) / df["puid"].nunique():.1%} от всех пользователей)')

display(df[df['puid'].isin(dup_puids)].sort_values('puid').head())

Пользователей, встречающихся в обеих группах: 244 (2.9% от всех пользователей)


,city,puid,hours
35,Москва,2637041,10.317371
6247,Санкт-Петербург,2637041,3.883926
134,Москва,9979490,32.415573
6274,Санкт-Петербург,9979490,1.302997
145,Москва,10597984,42.931506


Почти 3% пользователей (244 `puid`) встречаются одновременно в обеих группах — и в Москве, и в Санкт-Петербурге — с разными значениями hours. Скорее всего, это не техническая ошибка выгрузки, а пользователи, сменившие город (или использующие приложение в нескольких городах), либо не полностью нормализованные значения `city` в источнике. В любом случае одновременное присутствие одного и того же `puid` в обеих группах нарушает предпосылку о независимости выборок, необходимую для t-теста, поэтому такие записи нужно обработать до сравнения групп.

Удалим дубликаты,так как пользователи принадлежат разным городам, поэтому изначально нарушают предпосылку о независимости выборок

In [7]:
# Полностью удаляем пользователей, чей puid встречается в обеих группах
n_before = len(df)

dup_puids = df.loc[df['puid'].duplicated(keep=False), 'puid'].unique()
df = df[~df['puid'].isin(dup_puids)].copy()

print(f'Было строк: {n_before}, стало: {len(df)}')
print(f'Удалено строк (оба вхождения по каждому из {len(dup_puids)} дублирующихся puid): {n_before - len(df)}')
print('Проверка, что дублей по puid больше нет:', df['puid'].duplicated().sum())

Было строк: 8784, стало: 8296
Удалено строк (оба вхождения по каждому из 244 дублирующихся puid): 488
Проверка, что дублей по puid больше нет: 0


In [8]:
# Проверим наличие неявных дубликатов в городах
df['city'].unique()

array(['Москва', 'Санкт-Петербург'], dtype=object)

Итак, в данных нет пропусков и полных дубликатов, однако 244 пользователя (2,9%) встречаются одновременно в обеих группах — и в Москве, и в Санкт-Петербурге. Чтобы не нарушать независимость выборок, мы полностью удалили все записи этих пользователей — 488 строк. Осталось 8296 строк: 5990 пользователей из Москвы и 2306 из Санкт-Петербурга.

## Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [9]:
# Разбиваем пользователей на две группы(Москва и Санкт-Петербург)
msk = df.loc[df['city'] == 'Москва', 'hours']
spb = df.loc[df['city'] == 'Санкт-Петербург', 'hours']

In [10]:
# Описательные статистики
desc = pd.DataFrame({
    'Москва': msk.describe(),
    'Санкт-Петербург': spb.describe()
}).round(4)

print(desc)
print(f'Разница средних (СПб − Мск): {spb.mean() - msk.mean()} ч')

          Москва  Санкт-Петербург
count  5990.0000        2306.0000
mean     10.8482          11.2644
std      36.9256          39.8318
min       0.0000           0.0000
25%       0.0570           0.0602
50%       0.8882           0.8754
75%       5.9334           6.1384
max     857.2094         978.7648
Разница средних (СПб − Мск): 0.416241859890361 ч


In [11]:
# На каком уровне значимости проверяем гипотезу о равенстве выборочных средних
alpha = 0.05

In [12]:
stat_welch_ttest, p_value_welch_ttest = ttest_ind(
    spb,
    msk,
    equal_var=False,
    alternative='greater' 
)

In [13]:
if p_value_welch_ttest < alpha:
    print(f'p-value теста Уэлча = {round(p_value_welch_ttest, 4)}')
    print('Нулевая гипотеза отклоняется в пользу альтернативной')
    print('Пользователи из Санкт-Петербурга проводят статистически значимо больше времени '
          'за чтением и прослушиванием книг, чем пользователи из Москвы')
else:
    print(f'p-value теста Уэлча = {round(p_value_welch_ttest, 4)}')
    print('Нет оснований отклонить нулевую гипотезу')
    print('Различие в среднем времени активности пользователей из Санкт-Петербурга и Москвы статистически не значимо')

p-value теста Уэлча = 0.3318
Нет оснований отклонить нулевую гипотезу
Различие в среднем времени активности пользователей из Санкт-Петербурга и Москвы статистически не значимо


## Аналитическая записка


- **Тип теста и уровень значимости**. Для проверки гипотезы использован односторонний t-тест Уэлча (ttest_ind, equal_var=False, alternative='greater') для двух независимых выборок. Тест Уэлча выбран вместо классического теста Стьюдента, поскольку размеры групп существенно различаются (5990 пользователей из Москвы против 2306 из Санкт-Петербурга после полного удаления дублей), а дисперсии активности в группах также неравны (std ≈ 36.93 ч в Москве против 39.83 ч в СПб). Уровень статистической значимости: α = 0.05.  

- **Перед проверкой гипотезы данные были полностью очищены от дублирующихся пользователей**: 244 идентификатора (puid) встречались одновременно в обеих группах, что нарушало бы предпосылку о независимости выборок. Поскольку оставлять такого пользователя в одном из городов означало бы лишь замаскировать это нарушение, обе записи по каждому такому puid были удалены — всего 488 строк.  

- **Результат теста**. p-value = 0.3318. Средняя активность в Санкт-Петербурге (11.26 ч) выше, чем в Москве (10.85 ч), разница составляет 0.42 часа, однако при p-value = 0.3318 > α = 0.05 это различие статистически незначимо.  

- **Вывод**. Поскольку p-value превышает уровень значимости 0.05, у нас нет оснований отклонить нулевую гипотезу H0. Наблюдаемая разница в средней активности между городами (0.42 ч) не является статистически значимой и может объясняться случайной изменчивостью выборки, а не реальным различием в поведении пользователей. Данные не подтверждают гипотезу о том, что пользователи из Санкт-Петербурга в среднем проводят больше времени за чтением и прослушиванием книг, чем пользователи из Москвы.    


- **Возможные причины результата**. 
1. Распределение времени активности сильно скошено вправо (медиана активности в обеих группах меньше 1 часа, при этом среднее ≈ 11 часов) — поведение определяется в основном небольшой долей очень активных пользователей, а не городом проживания. Из-за этого городской фактор теряется на фоне огромного разброса значений внутри каждой группы (std в 3–4 раза больше среднего).
2. Размер выборки из Санкт-Петербурга почти в 2.6 раза меньше, чем из Москвы (2306 против 5990), что снижает статистическую мощность теста для этой группы и затрудняет обнаружение даже реально существующей, но небольшой разницы в поведении между городами.

# Часть 2. Анализ результатов A/B-тестирования

## Цели исследования


Цель исследования — оценить корректность проведения A/B-теста обновлённого интерфейса онлайн-магазина BitMotion Kit и проверить, приводит ли упрощённый интерфейс к росту конверсии зарегистрированных пользователей в покупателей в течение первых семи дней после регистрации не менее чем на 3 процентных пункта, как это предполагалось техническим заданием.

**План работ:**

1. Загрузить таблицы `ab_test_participants` и `ab_test_events`, проверить их на пропуски и дубликаты.
2. Выделить участников теста `interface_eu_test`, проверить равномерность распределения по группам и исключить пользователей, пересекающихся с конкурирующим тестом.
3. Определить горизонт анализа (7 дней с момента регистрации) и оставить только события в этом окне.
4. Оценить достаточность выборки для обнаружения ожидаемого эффекта.
5. Рассчитать конверсию по группам и проверить статистическую значимость различия подходящим тестом.
6. Сравнить фактический эффект с требованием технического задания и сформулировать итоговый вывод.

## Загрузка данных и проверка целостности


In [14]:
# Загружаем новые данные
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [15]:
# Знакомимся с полученными данными
print(participants.info())
print(events.info())
print('Пропуски в participants:\n', participants.isna().sum())
print('Полные дубли в participants:', participants.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB
None
Пропуски в participants:
 user_id    0
group      0
ab_test    0
device     0
dtype: int64
Полные дубли в participants: 0


## Проверка корректности проведения теста по таблице `ab_test_participants`

   3\.1 Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [16]:
test_name = 'interface_eu_test'
test_participants = participants[participants['ab_test'] == test_name].copy()

print(test_participants['group'].value_counts())
print(test_participants['group'].value_counts(normalize=True))

# Пользователи с конфликтующей группой внутри теста
conflict = test_participants.groupby('user_id')['group'].nunique()
print('Юзеров с конфликтом группы:', (conflict > 1).sum())

# Пересечение с конкурирующим тестом
other_test_users = set(participants.loc[participants['ab_test'] != test_name, 'user_id'])
overlap = test_participants[test_participants['user_id'].isin(other_test_users)]

print('Пересечение с recommender_system_test:', overlap['user_id'].nunique(),
      f"({overlap['user_id'].nunique() / test_participants['user_id'].nunique():.1%})")

clean_participants = test_participants[~test_participants['user_id'].isin(other_test_users)].copy()
print('Осталось после чистки:', clean_participants['group'].value_counts())

B    5467
A    5383
Name: group, dtype: int64
B    0.503871
A    0.496129
Name: group, dtype: float64
Юзеров с конфликтом группы: 0
Пересечение с recommender_system_test: 887 (8.2%)
Осталось после чистки: B    5011
A    4952
Name: group, dtype: int64


- Группы почти сбалансированы: A — 5383 (49.6%), B — 5467 (50.4%). 
- Конфликтующих по группе пользователей нет.
- 887 пользователей (8.2%) одновременно участвуют в recommender_system_test — это нарушает чистоту эксперимента (на них могло повлиять другое изменение), их нужно исключить. После чистки: A = 4952, B = 5011 — баланс групп практически не пострадал (исключили примерно поровну — 431 из A, 456 из B).

3\.2 Проанализируем данные о пользовательской активности по таблице `ab_test_events`:


In [17]:
test_user_ids = set(clean_participants['user_id'])
test_events = events[events['user_id'].isin(test_user_ids)].copy()

reg_events = test_events[test_events['event_name'] == 'registration'][['user_id', 'event_dt']]
reg_events = reg_events.rename(columns={'event_dt': 'reg_dt'})

# проверим,что у каждого участника ровно одна регистрация
print('Пользователей с регистрацией:', reg_events['user_id'].nunique(), 'из', len(clean_participants))
print('Дублей регистрации:', (reg_events.groupby('user_id').size() > 1).sum())

test_events = test_events.merge(reg_events, on='user_id', how='left')
test_events['lifetime'] = test_events['event_dt'] - test_events['reg_dt']

# Горизонт: последняя регистрация должна укладываться в 7 дней до конца данных
print('Диапазон регистраций:', reg_events['reg_dt'].min(), '—', reg_events['reg_dt'].max())
print('Конец данных:', events['event_dt'].max())

window_events = test_events[(test_events['lifetime'] >= pd.Timedelta(0)) & (test_events['lifetime'] <= pd.Timedelta(days=7))]
window_events = window_events.merge(clean_participants[['user_id', 'group']], on='user_id', how='left')

Пользователей с регистрацией: 9963 из 9963
Дублей регистрации: 0
Диапазон регистраций: 2020-12-06 14:10:01 — 2020-12-24 21:57:10
Конец данных: 2020-12-31 23:59:48


Результат: у всех 9963 участников есть ровно одна регистрация и хотя бы одно событие. Регистрации укладываются в промежуток 06.12–24.12.2020, а данные о событиях идут по 31.12.2020 включительно — значит, у всех пользователей 7-дневное окно наблюдения полностью попадает в собранные данные.

Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [18]:
p1, p2 = 0.30, 0.33 
alpha = 0.05
power = 0.8

effect_size = proportion_effectsize(p2, p1)

# Считаем нужный размер выборки на одну группу
n_required = zt_ind_solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1)

print(f'Нужно на группу: {n_required:.0f}')

Нужно на группу: 3762


Нужно не менее 3762 пользователей на группу. Фактически в группах 4952 и 5011 — выборки достаточно с запасом.

- Рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [19]:
summary = window_events.groupby('group').agg(visitors=('user_id', 'nunique'))
summary['purchasers'] = window_events[window_events['event_name'] == 'purchase'].groupby('group')['user_id'].nunique()
summary['conversion'] = summary['purchasers'] / summary['visitors']
display(summary)

,visitors,purchasers,conversion
group,,,
A,4952,1377,0.278069
B,5011,1480,0.295350


Предварительно, конверсия в B выше на 1.73 п.п

## Оценка результатов A/B-тестирования


- Проверим изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

In [20]:
n_A, n_B = summary.loc['A', 'visitors'], summary.loc['B', 'visitors']
x_A, x_B = summary.loc['A', 'purchasers'], summary.loc['B', 'purchasers']

# проверяем H1: конверсия B больше конверсии A
z_stat, p_value = proportions_ztest([x_B, x_A], [n_B, n_A], alternative='larger')

print(f'Конверсия A: {x_A/n_A:.2%}, конверсия B: {x_B/n_B:.2%}')
print(f'z = {z_stat:.4f}, p-value = {p_value:.4f}')

Конверсия A: 27.81%, конверсия B: 29.54%
z = 1.9070, p-value = 0.0283


Вывод: поскольку исходная гипотеза была односторонней ("упрощение интерфейса увеличит конверсию" — это H1: p_B > p_A), корректно использовать односторонний тест. При α = 0.05 он даёт p-value = 0.0283 < 0.05 — статистически рост конверсии в группе B подтверждается.

Но здесь важно разделить два разных вопроса. Статистическая значимость есть, но по ТЗ эффект должен быть не менее 3 процентных пунктов, а фактически наблюдается прирост всего 1.73 п.п. — почти вдвое меньше требуемого порога.

## Общий вывод

**Корректность теста**. Группы A и B изначально сбалансированы (49.6% / 50.4%). Обнаружено пересечение с конкурирующим тестом `recommender_system_test` — 887 пользователей (8.2%) участвовали в обоих тестах одновременно, их исключили из анализа (после чистки: A = 4952, B = 5011, баланс сохранился). Все пользователи зарегистрировались не позже 24 декабря, а данные собраны по 31 декабря — 7-дневное окно наблюдения полное у всех, без обрезки по краю периода.

**Достаточность выборки**. Для обнаружения ожидаемого эффекта в 3 п.п. (при базовой конверсии 30%, мощности 80%, α = 0.05) нужно не менее 3762 пользователей на группу. Фактический размер групп это требование перекрывает.

**Результат**. Конверсия в покупку за 7 дней: 27.81% в группе A, 29.54% в группе B — разница 1.73 п.п. в пользу нового интерфейса. Односторонний z-тест для двух пропорций (H1: конверсия B выше) дал z = 1.907, p-value = 0.028 — при α = 0.05 разница статистически значима.
Но по величине эффект не дотягивает до требования ТЗ. Задание требовало роста конверсии минимум на 3 п.п., а фактический прирост — 1.73 п.п., почти вдвое меньше. 


**Итог**. Эффект от нового интерфейса статистически значим — случайностью его не объяснить, — но по размеру (1.73 п.п.) он меньше заявленного в ТЗ порога в 3 п.п. Это расхождение между статистической и практической значимостью: гипотезу об улучшении можно принять, но цель теста в полном объёме не достигнута. Стоит либо продлить тест для более точной оценки эффекта, либо доработать интерфейс, чтобы усилить наметившийся положительный тренд.